In [3]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb
from rec_eval import (
    evaluate_model, build_popularity, relevant_from_val,
    recall_at_k, ndcg_at_k, reciprocal_rank, catalog_coverage, novelty,
)

# ------------------------------------------------------------------ config
MODEL_DIR = "../models"
DATA_DIR  = "../data"      # <-- adjust to wherever the CSVs live
N_SLOTS   = 6
K_LIST    = (5, 10, 20)
N_NEG     = 500            # negatives per user for the candidate pool
SEED      = 0
rng       = np.random.default_rng(SEED)

# ------------------------------------------------------------------ load models + bundles
def load_model_and_bundle(name):
    m = xgb.Booster()
    m.load_model(os.path.join(MODEL_DIR, f"{name}.json"))
    with open(os.path.join(MODEL_DIR, f"{name}_bundle.json")) as f:
        b = json.load(f)
    return m, b

model_slot,  bundle_slot  = load_model_and_bundle("model2_slot_reg")
model_deep,  bundle_deep  = load_model_and_bundle("model_slot_deep")
model_cand,  bundle_cand  = load_model_and_bundle("model_cand_only")

FEATURE_COLS = bundle_slot['feature_cols']
GAME_FEATS   = bundle_slot['game_feats']
CAND_ONLY    = bundle_cand['feature_cols']
print("slot feature cols:", len(FEATURE_COLS), "| game feats:", len(GAME_FEATS),
      "| cand-only:", len(CAND_ONLY))

# ------------------------------------------------------------------ load data
train = pd.read_csv(os.path.join(DATA_DIR, "training_data_train_final.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "training_data_val_final.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "training_data_test_final.csv"))

print("train:", train.shape, "| val:", val.shape, "| test:", test.shape)

# ------------------------------------------------------------------ catalog + popularity (from TRAIN only)
catalog_appids = train['appid'].unique()          # every game that appears in train
df_games_eval  = pd.DataFrame({'appid': catalog_appids})
popularity     = build_popularity(train)          # {appid: prob}

# users to evaluate = users present in val (fall back to test)
eval_users = val['steamid'].unique()
print("users to evaluate:", len(eval_users))

# ------------------------------------------------------------------ candidate pools
# For each user: their actual val games (positives) + N_NEG random catalog negatives.
def build_pools(eval_df, catalog_appids, n_neg, rng):
    pools = {}
    for uid, g in eval_df.groupby('steamid'):
        pos = g['appid'].tolist()
        pool = catalog_appids[~np.isin(catalog_appids, pos)]
        neg  = rng.choice(pool, size=min(n_neg, len(pool)), replace=False)
        pools[uid] = np.array(pos + list(neg))
    return pools

pools = build_pools(val, catalog_appids, N_NEG, rng)

# ground truth: {steamid: {appid: interest}} for the val games
relevant = relevant_from_val(val)

slot feature cols: 670 | game feats: 94 | cand-only: 94
train: (117489, 673) | val: (59850, 673) | test: (19787, 673)
users to evaluate: 20043


In [5]:
print(train.columns.tolist())

['steamid', 'appid', 'interest', 'slot1_n_labels', 'slot1_Action', 'slot1_Adventure', 'slot1_Casual', 'slot1_Simulation', 'slot1_Strategy', 'slot1_RPG', 'slot1_Sports', 'slot1_Racing', 'slot1_Massively Multiplayer', 'slot1_Shooter', 'slot1_FPS', 'slot1_Platformer', 'slot1_Puzzle', 'slot1_Exploration', 'slot1_Open World', 'slot1_Survival', 'slot1_Horror', 'slot1_Stealth', 'slot1_Fighting', 'slot1_Roguelike', 'slot1_Roguelite', 'slot1_Metroidvania', 'slot1_Souls-like', 'slot1_Turn-Based Combat', 'slot1_Turn-Based Strategy', 'slot1_RTS', 'slot1_Tower Defense', 'slot1_Card Game', 'slot1_Management', 'slot1_Building', 'slot1_Crafting', 'slot1_Sandbox', 'slot1_Visual Novel', 'slot1_Point & Click', 'slot1_Walking Simulator', 'slot1_Immersive Sim', 'slot1_Hack and Slash', 'slot1_Bullet Hell', 'slot1_Rhythm', 'slot1_Clicker', 'slot1_Auto Battler', 'slot1_MOBA', 'slot1_Battle Royale', 'slot1_Dungeon Crawler', 'slot1_Farming Sim', 'slot1_Life Sim', 'slot1_Grand Strategy', 'slot1_Parkour', 'slot1_

In [7]:
raw_games = pd.read_csv("../new_package/03_features/game_features.csv")
print(raw_games.shape)
print([c for c in raw_games.columns][:20])

(7430, 99)
['appid', 'name', 'inclusion', 'status', 'note', 'n_labels', 'Action', 'Adventure', 'Casual', 'Simulation', 'Strategy', 'RPG', 'Sports', 'Racing', 'Massively Multiplayer', 'Shooter', 'FPS', 'Platformer', 'Puzzle', 'Exploration']


In [9]:
# ------------------------------------------------------------------ game feature lookup
game_feats_df = raw_games.drop_duplicates('appid').set_index('appid')[GAME_FEATS]

# user's train games: appid + interest only
train_games_by_user = {
    uid: g.sort_values('interest', ascending=False)[['appid', 'interest']]
    for uid, g in train.groupby('steamid')
}

# ------------------------------------------------------------------ slot-row builder
def build_candidate_matrix(uid, candidate_appids):
    played = train_games_by_user.get(uid)
    feats  = game_feats_df.reindex(candidate_appids)
    n = len(candidate_appids)
    rows = {}

    if played is not None:
        played = played.head(N_SLOTS)
        for i in range(1, N_SLOTS + 1):
            for f in GAME_FEATS:
                rows[f'slot{i}_{f}'] = np.full(n, np.nan)
            rows[f'slot{i}_interest'] = np.full(n, np.nan)
            rows[f'slot{i}_present']  = np.zeros(n)
            if i <= len(played):
                g  = played.iloc[i - 1]
                gf = game_feats_df.loc[g['appid']]
                for f in GAME_FEATS:
                    rows[f'slot{i}_{f}'] = gf[f]
                rows[f'slot{i}_interest'] = g['interest']
                rows[f'slot{i}_present']  = 1

    for f in GAME_FEATS:
        rows[f'cand_{f}'] = feats[f].to_numpy()

    return pd.DataFrame(rows)


# ------------------------------------------------------------------ score functions
def make_score_slot(model, best_iter):
    def score_fn(uid, candidate_appids):
        X = build_candidate_matrix(uid, candidate_appids)
        return model.predict(xgb.DMatrix(X[FEATURE_COLS]),
                             iteration_range=(0, best_iter + 1))
    return score_fn

def make_score_cand(model, best_iter):
    def score_fn(uid, candidate_appids):
        feats = game_feats_df.reindex(candidate_appids)
        X = feats.to_numpy()
        Xdf = pd.DataFrame(X, columns=CAND_ONLY)
        return model.predict(xgb.DMatrix(Xdf),
                             iteration_range=(0, best_iter + 1))
    return score_fn

score_slot = make_score_slot(model_slot, bundle_slot['best_iteration'])
score_deep = make_score_slot(model_deep, bundle_deep['best_iteration'])
score_cand = make_score_cand(model_cand, bundle_cand['best_iteration'])

In [11]:
n_no_train = sum(1 for u in eval_users if u not in train_games_by_user)
print(f"eval users with no train games: {n_no_train} / {len(eval_users)}")

eval users with no train games: 0 / 20043


In [10]:
# ------------------------------------------------------------------ evaluate
def run_eval(score_fn, label):
    res = evaluate_model(
        users=eval_users,
        score_fn=score_fn,
        df_games=df_games_eval,
        relevant_by_user=relevant,
        item_popularity=popularity,
        candidates_per_user=pools,
        K=K_LIST,
    )
    res.insert(0, 'model', label)
    return res

results = pd.concat([
    run_eval(score_slot, 'slot_reg'),
    run_eval(score_deep, 'slot_deep'),
    run_eval(score_cand, 'cand_only'),
], ignore_index=True)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(results.to_string(index=False))

KeyboardInterrupt: 